# 02: CTC Baseline - Training Curves & Evaluation

This notebook evaluates the **CNN + BiLSTM + CTC** baseline model. It parses TensorBoard events to plot training curves, computes standard metrics (CER, WER, compound vs simple breakdown), and conducts an error analysis on predictions.

### Objectives:
1. **Load the trained CTC model** and run a mock evaluation (with fallback if no checkpoint exists).
2. **Plot learning curves** (loss and CER) from TensorBoard logs.
3. **Compute CER/WER** on the validation set.
4. **Analyze error distributions** (by word length and character classes).
5. **Visualize predictions** with side-by-side alignment.

## 1. Imports and Setup

In [ ]:
import os
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import editdistance

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from src.vocab import TeluguVocab, DEFAULT_VOCAB, VIRAMA
from src.models.ctc_model import CTCModel
from src.dataset import TeluguHTRDataset
from src.transforms import ValTransform

sns.set_theme(style="whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load Checkpoint or Create Mock Model

We attempt to load `checkpoints/ctc/best.pt`. If it does not exist, we construct a randomly initialized model and note that this is a placeholder.

In [ ]:
vocab_path = os.path.join("..", "checkpoints", "vocab.pkl")
ctc_ckpt_path = os.path.join("..", "checkpoints", "ctc", "best.pt")

# Load vocab
if os.path.exists(vocab_path):
    vocab = TeluguVocab.load(vocab_path)
else:
    print("vocab.pkl not found, using default permissive vocab")
    vocab = DEFAULT_VOCAB

vocab_size = len(vocab)
model = CTCModel(vocab_size=vocab_size, pretrained=False)

is_real_eval = False
if os.path.exists(ctc_ckpt_path):
    try:
        state = torch.load(ctc_ckpt_path, map_location="cpu")
        model.load_state_dict(state.get("model_state_dict", state))
        print(f"Successfully loaded real CTC checkpoint from {ctc_ckpt_path}")
        is_real_eval = True
    except Exception as e:
        print(f"Error loading checkpoint: {e}. Running in Mock Mode.")
else:
    print(f"Checkpoint not found at {ctc_ckpt_path}. Running in Mock Mode with dummy weights.")

model = model.to(device)
model.eval()

## 3. Extract and Plot Training Curves

We parse the TensorBoard event logs under `logs/ctc/`. If no logs are found, we generate synthetic training history to illustrate the training convergence.

In [ ]:
def get_training_history():
    # Attempt to read TB log files if package tensorboard is installed
    log_dir = os.path.join("..", "logs", "ctc")
    tb_files = glob.glob(os.path.join(log_dir, "**", "events.out.tfevents.*"), recursive=True)
    
    if tb_files:
        try:
            from tensorboard.backend.event_processing import event_accumulator
            print(f"Found TensorBoard log file: {tb_files[0]}")
            ea = event_accumulator.EventAccumulator(tb_files[0])
            ea.Reload()
            
            # Extract metrics
            train_loss = [e.value for e in ea.Scalars("train/loss")]
            val_loss = [e.value for e in ea.Scalars("val/loss")]
            val_cer = [e.value for e in ea.Scalars("val/CER")]
            epochs = list(range(1, len(train_loss) + 1))
            
            return pd.DataFrame({
                "epoch": epochs,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_cer": val_cer
            })
        except Exception as e:
            print("Failed to extract real TB logs:", e)
            
    # Generate synthetic training history representing standard CTC convergence
    print("Generating synthetic training history (fallback)...")
    epochs = np.arange(1, 31)
    # Loss decays exponentially
    train_loss = 4.5 * np.exp(-epochs/8) + 0.2 + np.random.normal(0, 0.05, 30)
    val_loss = 4.6 * np.exp(-epochs/9) + 0.35 + np.random.normal(0, 0.03, 30)
    # CER starts around 95% and converges to ~11%
    val_cer = 0.85 * np.exp(-epochs/6) + 0.11 + np.random.normal(0, 0.008, 30)
    # Clamp boundaries
    train_loss = np.clip(train_loss, 0.1, None)
    val_loss = np.clip(val_loss, 0.2, None)
    val_cer = np.clip(val_cer, 0.0, 1.0)
    
    return pd.DataFrame({
        "epoch": epochs,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_cer": val_cer
    })

history = get_training_history()

# Plot curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
sns.lineplot(data=history, x="epoch", y="train_loss", label="Train Loss", ax=axes[0], marker="o")
sns.lineplot(data=history, x="epoch", y="val_loss", label="Val Loss", ax=axes[0], marker="s")
axes[0].set_title("CTC Baseline: Training & Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Character Error Rate (CER)
sns.lineplot(data=history, x="epoch", y="val_cer", label="Val CER", ax=axes[1], color="crimson", marker="o")
axes[1].set_title("CTC Baseline: Validation Character Error Rate (CER)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("CER (lower is better)")
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Run Evaluation on Validation Set

We load the validation set. If we are running in Mock Mode, we generate simulated predictions (containing typical insertion/deletion/substitution errors) so that all subsequent evaluation metrics and graphs compile properly.

In [ ]:
val_ann_path = os.path.join("..", "data", "raw", "val", "labels.txt")
val_root = os.path.join("..", "data", "raw")

# Initialize validation dataset
if os.path.exists(val_ann_path):
    dataset = TeluguHTRDataset(val_ann_path, val_root, vocab, ValTransform(), add_sos_eos=False)
else:
    print("Validation label file not found. Creating mock dataset for evaluation...")
    # Fallback mock dataset
    dataset = None

# Run evaluation
eval_data = []

if dataset is not None and is_real_eval:
    # Run real evaluation
    print("Running real model inference over validation split...")
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)
    with torch.no_grad():
        for images, labels, lengths in loader:
            images = images.to(device)
            preds = model.greedy_decode(images)
            
            for i in range(len(preds)):
                gt = vocab.decode(labels[i][:lengths[i]].tolist(), strip_special=True)
                pred = preds[i]
                eval_data.append((gt, pred))
else:
    # Generate mock evaluation data
    print("Simulating validation predictions...")
    mock_words = [
        ("కాలం", "కాలం"),
        ("పూజ", "పూూజ"), # Vowel mark substitution/insertion
        ("తెలుగు", "తెలకు"), # Consonant substitution
        ("భారతదేశం", "భారతదేశ"), # Truncation / EOS deletion
        ("అమ్మ", "అమ"), # Virama/conjunct drop
        ("నాన్న", "నాన్న"),
        ("విద్యా", "విదా"), # Virama / ottu dropped
        ("ప్రగతి", "పరగతి"), # Stride alignment error
        ("సంస్కృతి", "సంసకృతి"),
        ("విశ్వవిద్యాలయం", "విశవవిదాలయం"),
        ("సూర్యుడు", "సూర్యుడు"),
        ("చంద్రుడు", "చద్రుడు"), # Virama consonant drop
        ("నక్షత్రం", "నక్షతరం"), # Ottu drop
        ("సముద్రం", "సముద్రం"),
        ("పర్వతం", "పరవతం")
    ]
    # Duplicate to expand samples
    for _ in range(5):
        eval_data.extend(mock_words)
        
df_eval = pd.DataFrame(eval_data, columns=["ground_truth", "prediction"])
# Compute edit distance metrics
def analyze_metrics(df):
    df = df.copy()
    df["distance"] = df.apply(lambda r: editdistance.eval(r["ground_truth"], r["prediction"]), axis=1)
    df["gt_len"] = df["ground_truth"].apply(len)
    df["pred_len"] = df["prediction"].apply(len)
    df["cer"] = df["distance"] / df["gt_len"]
    df["wer"] = (df["ground_truth"] != df["prediction"]).astype(float)
    
    # Check if contains virama (compound/conjunct character mark)
    df["is_compound"] = df["ground_truth"].apply(lambda s: VIRAMA in s)
    return df

df_results = analyze_metrics(df_eval)

mean_cer = df_results["cer"].mean() * 100
mean_wer = df_results["wer"].mean() * 100
print(f"Overall Val metrics: CER = {mean_cer:.2f}% | WER = {mean_wer:.2f}%")

## 5. Compound Character Breakdown

Handwritten Telugu text recognition experiences higher error rates on compound characters (which contain a Virama (్) to form conjunct ottus) compared to simple character strings. We isolate and analyze this breakdown below.

In [ ]:
compound_stats = df_results.groupby("is_compound").agg(
    count=("cer", "count"),
    mean_cer=("cer", lambda x: x.mean() * 100),
    mean_wer=("wer", lambda x: x.mean() * 100)
).reset_index()

compound_stats["Category"] = compound_stats["is_compound"].map({True: "Compound Words (Contains ్)", False: "Simple Words (No ్)"})
print(compound_stats.to_string(index=False))

# Plot compound vs simple CER
plt.figure(figsize=(8, 5))
sns.barplot(data=compound_stats, x="Category", y="mean_cer", palette="muted")
plt.title("CTC Baseline: Character Error Rate (CER) Comparison")
plt.ylabel("CER (%)")
plt.ylim(0, max(compound_stats["mean_cer"]) + 5)
for index, row in compound_stats.iterrows():
    plt.text(index, row['mean_cer'] + 1, f"{row['mean_cer']:.1f}%", color='black', ha="center", fontweight="bold")
plt.show()

## 6. Error Analysis: CER vs Word Length

Let's trace how the CTC baseline performs as words grow longer. Typically, CTC alignment struggles with long words because the encoder sequence stride output (S=64) has narrow margin space.

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_results, x="gt_len", y="cer", err_style="band", color="navy", marker="o")
plt.title("CTC Baseline: Character Error Rate (CER) vs Word Length")
plt.xlabel("Word Length (number of codepoints)")
plt.ylabel("Character Error Rate (CER)")
plt.ylim(-0.05, 1.05)
plt.show()

## 7. Sample Prediction Alignment

We list a side-by-side view of ground truth versus CTC baseline predictions to highlight typical failure modes.

In [ ]:
pd.set_option('display.max_colwidth', None)
sample_errors = df_results[df_results["cer"] > 0].sample(min(10, len(df_results)))
print(sample_errors[["ground_truth", "prediction", "cer", "is_compound"]].to_string(index=False))